# Qwen3-4B -- ViNumQA eval-only (dual-GPU inference)

Loads the already-trained LoRA adapter and evaluates it on `test.json`.
No training here -- reduced from
`vsf-qwen3-4b-sft-w-reasoning-eng-finqa-vinumqa.ipynb`, which is a combined
train+eval notebook where the training cells (`get_peft_model`,
`SFTTrainer`, `trainer.train()`, adapter save/merge/GGUF export) are all
skipped once a checkpoint already exists -- this notebook keeps only the
load-checkpoint -> inference -> score path, plus the dual-GPU inference
split that shortens the eval loop on Kaggle's two T4s.

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups (incl. Kaggle)
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install -q tabulate
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Load the trained adapter (no `get_peft_model` -- the checkpoint already has LoRA weights)

In [2]:
MAX_SEQ_LENGTH = 4567  # ViNumQA contexts (pre_text + table + post_text) can be long
ADAPTER_DIR = "/kaggle/input/datasets/ntphuc/qwen3-4b-sft-w-reasoning-eng-finqa-vinumqa/qwen3-4b-vinumqa-sft-adapter"

### Data prep -- test split only

No `train_df`/`valid_df` here since this notebook doesn't train.

In [3]:
import pandas as pd
from tabulate import tabulate

test_df = pd.read_json('/kaggle/input/datasets/ntphuc149x2/vlsp2025-vinumqa/test.json')
print(f"test={len(test_df)}")

test=497


In [ ]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

def process_split(df):
    df = df.copy()
    df["pre_text_processed"] = df.apply(formatting_pre_text, axis=1)
    df["post_text_processed"] = df.apply(formatting_post_text, axis=1)
    df["table_processed"] = df.apply(formatting_table, axis=1)
    df["table_raw"] = df["table"]  # keep raw rows for table_* row-name lookup at eval time
    df["input_question"] = df.apply(processing_input_question, axis=1)
    df["program_processed"] = df.apply(processing_program_content, axis=1)
    df["answer_processed"] = df.apply(processing_answer_content, axis=1)
    df = df[["id", "pre_text_processed", "table_processed", "table_raw", "post_text_processed", "input_question",
             "program_processed", "answer_processed"]]
    df.columns = ["id", "pre_text", "table", "table_raw", "post_text", "question", "program", "answer"]
    return df

test_df = process_split(test_df)
test_df["generated_program"] = ""
test_df.sample(n=3)


In [5]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### PROGRAM:"""

# Same prompt format as the 0-shot/1-shot/sft notebooks, so results are
# comparable across all experiments (only the training regime differs).


### Free GPU 0 before launching the dual-GPU workers

The `model`/`tokenizer` loaded above only exist to confirm the checkpoint
loads correctly -- inference itself happens in two separate subprocesses
(one per GPU) that each load their own copy of the adapter, so this
notebook-level model would otherwise sit on GPU 0 alongside worker 0's copy
for no reason.

### Dual-GPU inference

Splits `test_df` across Kaggle's two T4s. Each GPU gets its own OS process
with its own model copy (loaded fresh inside the worker) -- CUDA contexts
don't fork safely, and `multiprocessing`'s `spawn` start method can't pickle
a function defined inline in a notebook cell, so the worker is written out
as a standalone script and launched via `subprocess` instead. This also
rules out "copy the model to GPU 1 and infer on both": `model.to("cuda:1")`
*moves* weights rather than duplicating them, and even with two separate
model copies, calling `generate()` twice from a for-loop in one Python
process is still sequential -- true parallelism needs two OS processes.

Not using vLLM here: found unreliable on Kaggle T4s in an earlier session
(bf16 unsupported on compute capability 7.5, OOM from VRAM not fully
released across engine restarts within the same process).

In [ ]:
worker_script = r'''
import argparse
import gc
import json
import os
import sys

parser = argparse.ArgumentParser()
parser.add_argument("--gpu", type=int, required=True)
parser.add_argument("--input_json", type=str, required=True)
parser.add_argument("--output_json", type=str, required=True)
parser.add_argument("--adapter_dir", type=str, required=True)
parser.add_argument("--max_seq_length", type=int, required=True)
args = parser.parse_args()

os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu)  # must be set before importing torch/unsloth

import torch
from unsloth import FastLanguageModel

SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### PROGRAM:"""

QWEN3_THINK_END_TOKEN_ID = 151668  # "</think>"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=args.adapter_dir,
    max_seq_length=args.max_seq_length,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)
FastLanguageModel.for_inference(model)

rows = json.load(open(args.input_json, encoding="utf-8"))
results = {}
n_total = len(rows)

for i, row in enumerate(rows, start=1):
    prompt = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text"], table=row["table"],
        post_text=row["post_text"], question=row["question"],
    )
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs, max_new_tokens=1024,
            temperature=0.6, top_p=0.95, top_k=20, min_p=0,
        )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    try:
        idx = len(output_ids) - output_ids[::-1].index(QWEN3_THINK_END_TOKEN_ID)
    except ValueError:
        idx = 0

    # Keep the FULL raw decode too (skip_special_tokens=False, so </think>
    # -- or its absence -- is visible), not just the post-split halves. This
    # is what lets you manually review, after the run, exactly what the
    # model emitted and whether the </think> split point was found at all --
    # printing this mid-generate() from inside the worker doesn't work well
    # (each print gets interleaved/split across lines by the notebook's
    # per-line log reader), so it's saved to the JSON instead and reviewed
    # in the notebook after both GPUs finish.
    raw_full = tokenizer.decode(output_ids, skip_special_tokens=False)
    found_think_close = idx != 0 or "</think>" not in raw_full

    trace_text = tokenizer.decode(output_ids[:idx], skip_special_tokens=True).strip()
    program_text = tokenizer.decode(output_ids[idx:], skip_special_tokens=True).strip()

    results[str(row["index"])] = {
        "id": row["id"],
        "gold_program": row["gold_program"],
        "raw_output": raw_full,
        "found_think_close": found_think_close,
        "reasoning_trace": trace_text,
        "generated_program": program_text,
    }

    # No "[gpu N]" prefix here: the notebook process prefixes non-progress
    # lines itself, and matches bare "i/n_total" lines to compute a combined
    # total across both GPUs.
    print(f"{i}/{n_total}", flush=True)

    gc.collect()
    torch.cuda.empty_cache()

with open(args.output_json, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False)

print(f"[gpu {args.gpu}] done: {len(results)} samples", flush=True)
'''

with open("/kaggle/working/_dual_gpu_worker.py", "w", encoding="utf-8") as f:
    f.write(worker_script)

print("Worker script written to /kaggle/working/_dual_gpu_worker.py")

In [ ]:
import json
import re
import subprocess
import threading
import queue

_PROGRESS_LINE_RE = re.compile(r"^\d+/\d+$")  # matches worker's "i/n_total" progress lines only

half = len(test_df) // 2
chunks = [test_df.iloc[:half], test_df.iloc[half:]]

input_paths = []
output_paths = []
for i, chunk in enumerate(chunks):
    rows = [
        {
            "index": idx,
            "id": row["id"],
            "pre_text": row["pre_text"], "table": row["table"],
            "post_text": row["post_text"], "question": row["question"],
            "gold_program": row["program"], "gold_answer": row["answer"],
        }
        for idx, row in chunk.iterrows()
    ]
    in_path = f"/kaggle/working/_chunk_{i}_input.json"
    out_path = f"/kaggle/working/_chunk_{i}_output.json"
    with open(in_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False)
    input_paths.append(in_path)
    output_paths.append(out_path)

procs = []
for gpu_id, in_path, out_path in zip([0, 1], input_paths, output_paths):
    cmd = [
        "python", "/kaggle/working/_dual_gpu_worker.py",
        "--gpu", str(gpu_id),
        "--input_json", in_path,
        "--output_json", out_path,
        "--adapter_dir", ADAPTER_DIR,
        "--max_seq_length", str(MAX_SEQ_LENGTH),
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    procs.append(proc)

log_queue = queue.Queue()

def _reader(gpu_id, proc):
    for line in proc.stdout:
        log_queue.put((gpu_id, line.rstrip()))
    log_queue.put((gpu_id, None))  # sentinel: this process's stream is closed (process exited)

threads = [
    threading.Thread(target=_reader, args=(gpu_id, proc), daemon=True)
    for gpu_id, proc in zip([0, 1], procs)
]
for t in threads:
    t.start()

# Each worker only knows its own half's progress ("i/n_total" is local to
# that process); a running total across both GPUs has to be counted here,
# since this is the only place that sees both streams.
n_total_all = len(test_df)
n_done_all = 0
n_finished_streams = 0

while n_finished_streams < len(procs):
    gpu_id, line = log_queue.get()
    if line is None:
        n_finished_streams += 1
        continue
    if _PROGRESS_LINE_RE.match(line):
        n_done_all += 1
        print(f"TOTAL: {n_done_all}/{n_total_all}")
    else:
        print(f"[gpu {gpu_id}] {line}")  # errors, the final "done: N samples" still show

for proc in procs:
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Worker process failed with exit code {proc.returncode}")

# Each worker's output is {index_str: {"id", "gold_program", "raw_output",
# "found_think_close", "reasoning_trace", "generated_program"}}.
all_results = {}
for out_path in output_paths:
    with open(out_path, encoding="utf-8") as f:
        all_results.update(json.load(f))

for idx_str, r in all_results.items():
    test_df.at[int(idx_str), "generated_program"] = r["generated_program"]
    test_df.at[int(idx_str), "reasoning_trace"] = r["reasoning_trace"]
    test_df.at[int(idx_str), "raw_output"] = r["raw_output"]
    test_df.at[int(idx_str), "found_think_close"] = r["found_think_close"]

print(f"Done. {(test_df['generated_program'] != '').sum()} / {len(test_df)} samples generated.")
n_missing_think_close = (~test_df["found_think_close"]).sum()
print(f"Samples where </think> was NOT found in the raw output: {n_missing_think_close} / {len(test_df)}")

### Manual review

Prints every sample where `</think>` wasn't found in the raw output first
(these are the ones whose `generated_program` is likely to contain leftover
reasoning-trace prose instead of a clean program string), then a handful of
normal ones for comparison. `raw_output` is decoded with
`skip_special_tokens=False`, so the literal `<think>`/`</think>` tokens (or
their absence) are visible exactly as the model produced them.

In [ ]:
def print_sample(idx, row):
    print(f"=== row {idx} (id={row['id']}) found_think_close={row['found_think_close']} ===")
    print("QUESTION:", row["question"])
    print("GOLD PROGRAM:", row["program"])
    print()
    print("RAW OUTPUT (skip_special_tokens=False):")
    print(row["raw_output"])
    print()
    print("PARSED generated_program:", repr(row["generated_program"]))
    print("=" * 100)
    print()

missing = test_df[~test_df["found_think_close"]]
print(f"{len(missing)} samples with no </think> found -- reviewing all of them:\n")
for idx, row in missing.iterrows():
    print_sample(idx, row)

# A handful of normal ones too, for comparison.
print("\n\n--- 5 samples where </think> WAS found, for comparison ---\n")
normal = test_df[test_df["found_think_close"]].head(5)
for idx, row in normal.iterrows():
    print_sample(idx, row)

In [ ]:
test_df

### PA / EA on the ViNumQA test set

Same scorer as every other prompting/SFT notebook in this repo
(`notebooks/evaluate/scorer.py`), inlined here so this notebook has no
external file dependency on Kaggle.

In [ ]:
"""ViNumQA scorer: the FinQA evaluation protocol, adapted to this dataset.

The shared-task paper states that "the official evaluation protocol proposed by
Chen et al. (2021) is adopted", so the semantics here follow `evaluate/evaluate.py`
(FinQA's own script) rather than being reinvented:

  * Program Accuracy is *symbolic* equivalence, via sympy, between the gold and
    predicted expressions -- not a string or structural match. A prediction may
    reorder or restructure the arithmetic, but it may only use literals that
    appear in the gold program, so it cannot invent constants such as the `100`
    of a percentage rescaling.
  * Execution Accuracy compares the executed result to `exe_ans` exactly, after
    rounding to 5 decimals. No tolerance.
  * `greater` yields the strings "yes"/"no", matching how the dataset stores
    those answers.
  * Every step takes exactly two arguments. Verified against the data: all 663
    steps across the gold programs are binary, and `table_*` always takes a row
    label plus `none` (454 occurrences) rather than a list of values (2).
  * FinQA's `const_` tokens are still understood.

Five corrections are applied, each because the unmodified script cannot
reproduce ViNumQA's own gold, not because the protocol was thought wrong:

1. Tokenisation of bracketed row labels. `program_tokenization` splits on every
   bracket, so `table_min(ROE (%), none)` shatters into six tokens and fails the
   four-tokens-per-step structure check. 35 of the 497 test programs name a row
   whose label contains brackets -- `ROE (%)`, `EPS (VND)`, `P/E (x)` -- and all
   35 were unscoreable. Tokenisation is now bracket-depth aware.

2. Accounting negatives. Tables write negative amounts as `(3344)`. The original
   `process_row` takes the text before the first bracket, leaving an empty
   string, so the cell fails to parse. The dataset's own `exe_ans` was computed
   with those values -- e.g. `table_min(LN hoạt động (tỷ đồng), none)` expects
   -3344 from a row holding `(3344)`. The `-1046 ( 1046 )` form the original
   handled correctly is unchanged.

3. Unparseable cells no longer void the whole row. Measured over the 393 gold
   `table_*(<row>, none)` programs in train, skipping such cells reproduces
   `exe_ans` for 386 against 381 when the row is voided, so skipping is what the
   dataset was built with.

4. `exe_ans` is stored as a string here ("31.0") where FinQA stores a number, so
   the comparison `exe_res == gold_res` was never true. It is coerced, leaving
   the "yes"/"no" answers alone.

5. The `assert exe_res == gold_res` inside the program-accuracy branch is
   dropped. It is a debug check, and a single rounding disagreement aborts the
   whole evaluation.

`evaluate_result_official` runs the unmodified protocol for comparison, so the
cost of each correction can be seen rather than assumed.
"""

import re
from typing import List, Optional, Sequence, Tuple, Union

from sympy import simplify

ALL_OPS = ["add", "subtract", "multiply", "divide", "exp", "greater",
           "table_max", "table_min", "table_sum", "table_average"]

_PAREN_NEG_RE = re.compile(r"^\(\s*([\d.,]+)\s*\)$")
_NAME_RE = re.compile(r"\s*([a-zA-Z_]+)\(")


# ------------------------------------------------------------------ numbers --
def str_to_num(text: str) -> Union[float, str]:
    """FinQA's literal parser, unchanged: returns "n/a" rather than raising."""
    text = str(text).replace(",", "")
    try:
        return float(text)
    except ValueError:
        if "%" in text:
            try:
                return float(text.replace("%", "")) / 100.0
            except ValueError:
                return "n/a"
        if text.endswith(("x", "X")):
            try:
                return float(text[:-1])
            except ValueError:
                pass
        if "const" in text:
            text = text.replace("const_", "")
            if text == "m1":
                text = "-1"
            try:
                return float(text)
            except ValueError:
                return "n/a"
        return "n/a"


def _cell_to_num(raw: str) -> Union[float, str]:
    """Parse one table cell.

    Adds the `(3344)` form to what the original handled; `$ -1046 ( 1046 )`
    still resolves through the original's "text before the first bracket" rule.
    """
    text = str(raw).replace("$", "").strip()
    m = _PAREN_NEG_RE.match(text)
    if m:
        value = str_to_num(m.group(1))
        return -value if value != "n/a" else "n/a"
    return str_to_num(text.split("(")[0].strip())


_MISSING_CELL_MARKERS = {"", "-", "–", "—", "na", "n/a", "nan", "none"}


def process_row(row_in: Sequence[str]):
    """Numeric values of a table row, or "n/a" if the row cannot be reduced."""
    row_out = []
    for cell in row_in:
        text = str(cell).replace("$", "").strip()
        if text.lower() in _MISSING_CELL_MARKERS:
            continue
        num = _cell_to_num(text)
        if num == "n/a":
            return "n/a"
        row_out.append(num)
    return row_out or "n/a"


# -------------------------------------------------------------- tokenisation --
def program_tokenization(original_program: str) -> List[str]:
    """Tokenise into ['op(', arg1, arg2, ')', ..., 'EOF']. Bracket-depth aware."""
    text = str(original_program).strip()
    program: List[str] = []
    pos = 0

    while pos < len(text):
        m = _NAME_RE.match(text, pos)
        if not m:
            break
        open_idx = m.end() - 1

        depth, close = 0, -1
        for i in range(open_idx, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            raise ValueError(
                f"Unbalanced parentheses (no matching \')\' found) in program: \'{original_program}\'"
            )

        program.append(m.group(1) + "(")
        args, arg_depth, current = [], 0, []
        for ch in text[m.end():close]:
            if ch == "(":
                arg_depth += 1
                current.append(ch)
            elif ch == ")":
                arg_depth -= 1
                current.append(ch)
            elif ch == "," and arg_depth == 0:
                args.append("".join(current).strip())
                current = []
            else:
                current.append(ch)
        if current:
            args.append("".join(current).strip())

        program.extend(args)
        program.append(")")
        pos = close + 1
        while pos < len(text) and text[pos] in ", ":
            pos += 1

    if pos < len(text) and text[pos:].strip():
        raise ValueError(
            f"Trailing unparsed content in program: \'{text[pos:]}\' (from: \'{original_program}\')"
        )

    program.append("EOF")
    return program


def extract_program(raw_text: str) -> str:
    """Recover a program string from raw model output. Bracket matched."""
    text = re.sub(r"```[a-zA-Z]*", "", str(raw_text)).replace("```", "").strip()

    calls, pos = [], 0
    while pos < len(text):
        m = _NAME_RE.search(text, pos)
        if not m:
            break
        if m.group(1) not in ALL_OPS:
            pos = m.end()
            continue
        depth, close = 0, -1
        for i in range(m.end() - 1, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            break
        calls.append(text[m.start(1):close + 1].strip())
        pos = close + 1

    return ", ".join(calls) if calls else text


def _steps_from_tokens(program: List[str]) -> List[Tuple[str, str, str]]:
    """Group a tokenised program into (op, arg1, arg2) triples."""
    body = program[:-1] if program and program[-1] == "EOF" else list(program)
    if len(body) % 4 != 0:
        raise ValueError("token count is not a multiple of four")
    steps = []
    for i in range(0, len(body), 4):
        op_token, arg1, arg2, close = body[i:i + 4]
        if not op_token.endswith("(") or close != ")":
            raise ValueError("malformed step")
        op = op_token[:-1].strip()
        if op not in ALL_OPS:
            raise ValueError(f"unknown operator {op!r}")
        steps.append((op, arg1.strip(), arg2.strip()))
    return steps

# ---------------------------------------------------------------- execution --
def eval_program(program: List[str], table: Optional[Sequence[Sequence[str]]]):
    """Execute a tokenised program. Returns (invalid_flag, result)."""
    this_res: Union[float, str] = "n/a"

    try:
        steps = _steps_from_tokens(program)
        res_dict = {}

        for ind, (op, arg1, arg2) in enumerate(steps):
            if op in ("add", "subtract", "multiply", "divide", "exp", "greater"):
                if "#" in arg1:
                    arg1 = res_dict[int(arg1.replace("#", ""))]
                else:
                    arg1 = str_to_num(arg1)
                    if arg1 == "n/a":
                        return 1, "n/a"
                if "#" in arg2:
                    arg2 = res_dict[int(arg2.replace("#", ""))]
                else:
                    arg2 = str_to_num(arg2)
                    if arg2 == "n/a":
                        return 1, "n/a"

                if op == "add":
                    this_res = arg1 + arg2
                elif op == "subtract":
                    this_res = arg1 - arg2
                elif op == "multiply":
                    this_res = arg1 * arg2
                elif op == "divide":
                    this_res = arg1 / arg2
                elif op == "exp":
                    this_res = arg1 ** arg2
                else:
                    this_res = "yes" if arg1 > arg2 else "no"

            else:  # table_*
                table_dict = {row[0]: row[1:] for row in (table or [])}
                if "#" in arg1:
                    num_row = [res_dict[int(arg1.replace("#", ""))]]
                else:
                    if arg1 not in table_dict:
                        return 1, "n/a"
                    num_row = process_row(table_dict[arg1])
                if num_row == "n/a":
                    return 1, "n/a"

                if op == "table_max":
                    this_res = max(num_row)
                elif op == "table_min":
                    this_res = min(num_row)
                elif op == "table_sum":
                    this_res = sum(num_row)
                else:
                    this_res = sum(num_row) / len(num_row)

            res_dict[ind] = this_res

        if this_res not in ("yes", "no", "n/a"):
            this_res = round(this_res, 5)
    except Exception:
        return 1, "n/a"

    return 0, this_res


# ------------------------------------------------------------------ program --
def equal_program(program1: List[str], program2: List[str]) -> bool:
    """Symbolic equivalence of gold (program1) and prediction (program2)."""
    try:
        steps1 = _steps_from_tokens(program1)
    except Exception:
        return False

    sym_map, sym_ind = {}, 0
    for op, arg1, arg2 in steps1:
        if "table" in op:
            key = (op, arg1, arg2)
            if key not in sym_map:
                sym_map[key] = "a" + str(sym_ind)
                sym_ind += 1
        else:
            for arg in (arg1, arg2):
                if "#" not in arg and arg not in sym_map:
                    sym_map[arg] = "a" + str(sym_ind)
                    sym_ind += 1

    try:
        steps2 = _steps_from_tokens(program2)
    except Exception:
        return False

    for ind, (op, arg1, arg2) in enumerate(steps2):
        if "table" in op:
            if (op, arg1, arg2) not in sym_map:
                return False
        else:
            for arg in (arg1, arg2):
                if "#" not in arg:
                    if arg not in sym_map:
                        return False
                elif int(arg.strip("#")) >= ind:
                    return False

    def symbol_recur(ind, steps):
        op, arg1, arg2 = steps[ind]
        if "table" in op:
            return sym_map[(op, arg1, arg2)]
        parts = []
        for arg in (arg1, arg2):
            if "#" in arg:
                parts.append(symbol_recur(int(arg.replace("#", "")), steps))
            else:
                parts.append(sym_map[arg])
        sign = {"add": "+", "subtract": "-", "multiply": "*",
                "divide": "/", "exp": "**", "greater": ">"}[op]
        return f"( {parts[0]} {sign} {parts[1]} )"

    try:
        sym1 = simplify(symbol_recur(len(steps1) - 1, steps1), evaluate=False)
        sym2 = simplify(symbol_recur(len(steps2) - 1, steps2), evaluate=False)
    except Exception:
        return False

    return sym1 == sym2


# ------------------------------------------------------------------ metrics --
def _coerce_answer(value):
    """ViNumQA stores exe_ans as a string; "yes"/"no" stay as they are."""
    try:
        return float(value)
    except (TypeError, ValueError):
        return value


def score_one(generated_program: str, gold_program: str, gold_answer,
              table: Optional[Sequence[Sequence[str]]] = None,
              extract_first: bool = True) -> Tuple[float, float]:
    """(program_accuracy, execution_accuracy) for a single item."""
    generated = extract_program(generated_program) if extract_first else generated_program
    gold_tok = program_tokenization(gold_program)
    gold_res = _coerce_answer(gold_answer)

    try:
        pred_tok = program_tokenization(generated)
    except ValueError:
        return 0.0, 0.0

    invalid, exe_res = eval_program(pred_tok, table)
    ea = 1.0 if invalid == 0 and exe_res == gold_res else 0.0

    try:
        pa = 1.0 if equal_program(gold_tok, pred_tok) else 0.0
    except Exception:
        pa = 0.0

    return pa, ea


def evaluate_dataframe(df, generated_col: str = "generated_program",
                       gold_program_col: str = "program",
                       gold_answer_col: str = "answer",
                       table_col: str = "table_raw",
                       extract_first: bool = True):
    """Score a DataFrame, returning (df + per-row scores, summary)."""
    df = df.copy()
    pa_scores, ea_scores = [], []

    for _, row in df.iterrows():
        table = row[table_col] if table_col in df.columns else None
        pa, ea = score_one(row[generated_col], row[gold_program_col],
                           row[gold_answer_col], table, extract_first)
        pa_scores.append(pa)
        ea_scores.append(ea)

    df["pa_score"] = pa_scores
    df["ea_score"] = ea_scores
    return df, {
        "program_accuracy": sum(pa_scores) / len(pa_scores) if pa_scores else 0.0,
        "execution_accuracy": sum(ea_scores) / len(ea_scores) if ea_scores else 0.0,
    }


In [ ]:
df_scored, summary = evaluate_dataframe(
    test_df,
    generated_col="generated_program",
    gold_program_col="program",
    gold_answer_col="answer",
    table_col="table_raw",
)

print(summary)  # {'program_accuracy': ..., 'execution_accuracy': ...}
